In [ ]:
import sys
sys.path.append("..")

from src.data_loader import df

df.head()

In [ ]:
%pip install kaggle

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2)

In [ ]:
df.duplicated().sum()

In [ ]:
df["length_of_encounter_seconds"].head(20)

In [ ]:
import pandas as pd

pd.to_numeric(df["length_of_encounter_seconds"], errors="coerce").isna().sum()

In [ ]:
numeric_duration = pd.to_numeric(
    df["length_of_encounter_seconds"],
    errors="coerce"
)

df[numeric_duration.isna()][
    ["length_of_encounter_seconds", "described_duration_of_encounter"]
]

In [ ]:
df["length_of_encounter_seconds"] = pd.to_numeric(
    df["length_of_encounter_seconds"].str.replace("`", "", regex=False),
    errors="coerce"
)

df["length_of_encounter_seconds"].dtype

In [ ]:
df["length_of_encounter_seconds"].isna().sum()

In [ ]:
df["length_of_encounter_seconds"].isna().sum()

In [ ]:
df["length_of_encounter_seconds"] = pd.to_numeric(
    df["length_of_encounter_seconds"].astype(str).str.replace("`", "", regex=False),
    errors="coerce"
)

df["length_of_encounter_seconds"].isna().sum()

In [ ]:
test_duration = pd.to_numeric(
    df["length_of_encounter_seconds"].astype(str).str.replace("`", "", regex=False),
    errors="coerce"
)

df.loc[test_duration.isna(), "length_of_encounter_seconds"].head(30)

In [ ]:
df["length_of_encounter_seconds"].isna().sum()

In [ ]:
import pandas as pd

In [ ]:
df["duration_seconds_clean"] = pd.to_numeric(
    df["length_of_encounter_seconds"]
      .astype(str)
      .str.replace("`", "", regex=False),
    errors="coerce"
)

df["duration_seconds_clean"].isna().sum()

In [ ]:
pd.to_numeric(df["latitude"], errors="coerce").isna().sum()

In [ ]:
latitude_numeric = pd.to_numeric(df["latitude"], errors="coerce")

df[latitude_numeric.isna()][
    ["latitude", "longitude", "city", "country"]
]

In [ ]:
df["latitude_clean"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["latitude_clean"].isna().sum()

In [ ]:
df["UFO_shape"].value_counts().head(10)

In [ ]:
df["datetime_clean"] = pd.to_datetime(
    df["Date_time"],
    errors="coerce"
)

df["datetime_clean"].isna().sum()

In [ ]:
df.loc[df["datetime_clean"].isna(), "Date_time"].head(20)

In [ ]:
mask_24 = df["Date_time"].str.endswith("24:00")

dates_24 = pd.to_datetime(
    df.loc[mask_24, "Date_time"].str.replace("24:00", "00:00", regex=False),
    errors="coerce"
) + pd.Timedelta(days=1)

df.loc[mask_24, "datetime_clean"] = dates_24

df["datetime_clean"].isna().sum()

In [ ]:
df["year"] = df["datetime_clean"].dt.year
df["month"] = df["datetime_clean"].dt.month
df["day_of_week"] = df["datetime_clean"].dt.day_name()
df["hour"] = df["datetime_clean"].dt.hour

df[["datetime_clean", "year", "month", "day_of_week", "hour"]].head()

In [ ]:
sightings_per_year = df["year"].value_counts().sort_index()

sightings_per_year.tail(20)

In [ ]:
df["datetime_clean"].min(), df["datetime_clean"].max()

In [ ]:
df.nsmallest(10, "datetime_clean")[
    ["Date_time", "datetime_clean", "city", "country", "description"]
]

In [ ]:
df["hour"].value_counts().sort_index()

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

df["day_of_week"].value_counts().reindex(day_order)

In [ ]:
import pandas as pd

In [ ]:
population_df = pd.read_csv(
    "../data/US-population-by-state(wide)-selected-columns.csv"
)

population_df.head()

In [ ]:
population_df.shape

In [ ]:
population_df.columns

In [ ]:
df["state/province"].value_counts().head(20)

In [ ]:
states_to_check = ["al", "ak", "az", "ar", "ca", "co", "ct", "de", "dc"]

df[df["state/province"].isin(states_to_check)]["state/province"].value_counts()

In [ ]:
population_df["year"].min(), population_df["year"].max()

In [ ]:
population_long = population_df.melt(
    id_vars="year",
    var_name="state",
    value_name="population"
)

population_long.head(10)

In [ ]:
population_long[
    population_long["year"].between(1906, 2014)
]["population"].isna().sum()

In [ ]:
population_long[
    population_long["population"].notna()
]["year"].unique()

In [ ]:
population_long[
    population_long["year"].between(1906, 2014)
].groupby("state")["population"].apply(lambda x: x.isna().sum())

In [ ]:
population_long[
    (population_long["state"] == "Alaska") &
    (population_long["population"].notna())
][["year", "population"]].head(10)

In [ ]:
population_long[
    population_long["year"] == 2014
][["state", "population"]]

In [ ]:
state_mapping = {
    "al": "Alabama",
    "ak": "Alaska",
    "az": "Arizona",
    "ar": "Arkansas",
    "ca": "California",
    "co": "Colorado",
    "ct": "Connecticut",
    "de": "Delaware",
    "dc": "District of Columbia"
}

In [ ]:
df["state"] = df["state/province"].map(state_mapping)

df[["state/province", "state"]].dropna().head(10)

In [ ]:
df["state"].notna().sum()

In [ ]:
ufo_per_state_year = (
    df[df["state"].notna()]
    .groupby(["year", "state"])
    .size()
    .reset_index(name="ufo_count")
)

ufo_per_state_year.head(10)

In [ ]:
ufo_per_state_year.shape, population_long.shape

In [ ]:
merged_df = ufo_per_state_year.merge(
    population_long,
    on=["year", "state"],
    how="left"
)

merged_df.shape

In [ ]:
merged_df[merged_df["population"].isna()][
    ["year", "state", "ufo_count", "population"]
]

In [ ]:
analysis_df = merged_df.dropna(subset=["population"]).copy()

analysis_df.shape

In [ ]:
analysis_df["ufo_per_100k"] = (
    analysis_df["ufo_count"] / analysis_df["population"]
) * 100000

analysis_df.head(10)

In [ ]:
analysis_df[
    analysis_df["year"] == 2013
].sort_values("ufo_per_100k", ascending=False)

In [ ]:
analysis_df["ufo_per_100k"].describe()

In [ ]:
analysis_df.loc[analysis_df["ufo_per_100k"].idxmax()]

In [ ]:
yearly_sightings = (
    df[df["year"].between(1949, 2013)]
    .groupby("year")
    .size()
    .reset_index(name="ufo_count")
)

yearly_sightings.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    yearly_sightings["year"],
    yearly_sightings["ufo_count"]
)

plt.xlabel("Jaar")
plt.ylabel("Aantal UFO-meldingen")
plt.title("Aantal geregistreerde UFO-meldingen per jaar (1949–2013)")

plt.show()

In [ ]:
state_2013 = (
    analysis_df[analysis_df["year"] == 2013]
    .sort_values("ufo_per_100k", ascending=False)
)

state_2013

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    state_2013["state"],
    state_2013["ufo_per_100k"]
)

plt.xlabel("Staat")
plt.ylabel("UFO-meldingen per 100.000 inwoners")
plt.title("UFO-meldingen per 100.000 inwoners per staat (2013)")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
hourly_sightings = (
    df["hour"]
    .value_counts()
    .sort_index()
    .reset_index()
)

hourly_sightings.columns = ["hour", "ufo_count"]

hourly_sightings

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    hourly_sightings["hour"],
    hourly_sightings["ufo_count"],
    marker="o"
)

plt.xlabel("Uur van de dag")
plt.ylabel("Aantal UFO-meldingen")
plt.title("Aantal geregistreerde UFO-meldingen per uur")

plt.xticks(range(0, 24))
plt.grid(alpha=0.3)

plt.show()

In [ ]:
shape_counts = (
    df["UFO_shape"]
    .value_counts()
    .head(10)
    .reset_index()
)

shape_counts.columns = ["UFO_shape", "ufo_count"]

shape_counts

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    shape_counts["UFO_shape"],
    shape_counts["ufo_count"]
)

plt.xlabel("Aantal UFO-meldingen")
plt.ylabel("UFO-vorm")
plt.title("Top 10 meest geregistreerde UFO-vormen")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()

In [ ]:
df.columns

In [ ]:
analysis_df.info()

In [ ]:
analysis_df = analysis_df.reset_index(drop=True)

analysis_df.info()

In [ ]:
population_long[
    population_long["state"] != "Alaska"
].groupby("year")["population"].count().tail(30)

In [ ]:
population_analysis = population_long[
    (population_long["state"] != "Alaska") &
    (population_long["year"].between(1949, 2013))
].copy()

population_analysis.shape

In [ ]:
dashboard_df = population_analysis.merge(
    ufo_per_state_year,
    on=["year", "state"],
    how="left"
)

dashboard_df["ufo_count"] = dashboard_df["ufo_count"].fillna(0).astype(int)

dashboard_df.shape

In [ ]:
dashboard_df["ufo_per_100k"] = (
    dashboard_df["ufo_count"] / dashboard_df["population"]
) * 100000

dashboard_df.head()

In [ ]:
import streamlit as st

st.__version__

In [ ]:
pdf = pd.read_csv("ufo_sighting_data.csv")
print(pdf.head())